<a href="https://colab.research.google.com/github/PeteH-89/GEOG5003M_Project/blob/main/GEOG5003M_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setting up**

**Begin by installing packages, connecting to Google Drive, and importing required packages**

In [ ]:
#uncomment to install mapclassify if required
!pip install mapclassify

#uncomment to connect to drive if required
from google.colab import drive
drive.mount('/content/drive')

# import required packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
import geopandas as gpd
from sklearn import cluster
from sklearn.preprocessing import scale

warnings.filterwarnings("ignore")

**Read in Data and Filter LSOAs to Only North Northamptonshire**

Using LSOA data from the Open Geography Portal (https://geoportal.statistics.gov.uk/) and Census data from https://www.nomisweb.co.uk/, read in and then use .loc to filter LSOA data to only North Northants.

In [ ]:
#read in datasets from drive
edu=pd.read_csv("/content/drive/MyDrive/GEOG5003M_Project/nnc_education.csv")
print(type(edu))
imd=pd.read_csv("/content/drive/MyDrive/GEOG5003M_Project/nnc_imd.csv")
lsoa_shp=gpd.read_file("/content/drive/MyDrive/GEOG5003M_Project/ew_lsoa.geojson")

In [ ]:
#use .head() to work out which column to filter on and then use .loc to filter lsoa data to only those within North Northamptonshire, finish by double checking everything is there using .explore
lsoa_shp.head()
nnc_lsoa=lsoa_shp.loc[lsoa_shp["LSOA21NM"].str.contains("North Northamptonshire")]
nnc_lsoa.to_file("nnc_lsoa.geojson", driver="GeoJSON")
nnc_lsoa.explore()

# **Data Exploration and Cleansing**

Check out the data we have downloaded - Look for any NAs, drop any unnecessary columns and do some basic checks on distributions etc.

In [ ]:
#start by checking dimensions of datasets, look for NAs and if present determine what to do with them (as using ONS data this is unlikely to be a problem)
print(edu.info())
print(edu.isna().sum())
print(imd.info())
print(imd.isna().sum())
print(nnc_lsoa.info())
print(nnc_lsoa.isna().sum())

All of the dataframes have 205 entries and 0 NAs, this is a good start, but the column names in the education and deprivation data need some adjustments to be usable.

In [ ]:
#rename the columns in edu and imd dataframes
edu.columns=['lsoa_name', 'lsoacode', 'pop', 'pop_pct', 'noqual_raw', 'noqual_pct', 'l1_raw', 'l1_pct', 'l2_raw', 'l2_pct', 'apprenticeship', 'apprenticeship_pct', 'l3_raw', 'l3_pct', 'l4_raw', 'l4_pct', 'other_raw', 'other_pct']
imd.columns=['lsoa_name', 'lsoacode', 'households', 'pop_pct', 'nodep_raw', 'nodep_pct', '1dim_raw', '1dim_pct', '2dim_raw', '2dim_pct', '3dim_raw', '3dim_pct', '4dim_raw', '4dim_pct']
#define as new dataframes for only percentage data with lsoa_name dropped as this is effectively duplicated by LSOA Code and only one is needed to join to geojson, pop_pct columns dropped as these are 100%. Raw population may remain useful however.
edupct=edu.drop(['lsoa_name', 'pop_pct', 'noqual_raw', 'l1_raw', 'l2_raw', 'apprenticeship', 'l3_raw', 'l4_raw', 'other_raw'], axis=1)
deppct=imd.drop(['lsoa_name', 'pop_pct', 'nodep_raw', '1dim_raw', '2dim_raw', '3dim_raw', '4dim_raw'], axis=1)
#join the two percentage tables and define as new dataframe.
edu_dep=pd.DataFrame(pd.merge(left=edupct, right=deppct, how='left', on='lsoacode'))
print(edu_dep.info())
edu_dep.describe()

The data description appears to be fairly reasonable - Our area has a blend of rural areas and small and mid sized towns and that would reasonably account for the variation between LSOAs. There is also a prison and a boarding school which is the likely reason for populations at the higher end of the distribution. One notable thing is that there are certain LSOAs without any severe (3 or 4 dimensions) deprivation. Splitting LSOAs into deciles based on percentage of households in each set of dimensions may be helpful for analysis.

In [ ]:
#using a for loop we can add the deciles only for the deprivation deciles, then we can map the deprivation deciles and see if the more deprived areas are where we would expect them to be.
#start by defining a list of the columns we want to create deciles for
#dep_cols=['nodep_pct', '1dim_pct','2dim_pct', '3dim_pct', '4dim_pct']
#now use a for loop to calculate a new column for each dep_col
#for col in dep_cols:
  #replace "pct" in new column names with "dec"
  #dec_col=col.replace('_pct', '_dec')
  #calculate 10 bins using qcut
  #edu_dep[dec_col]=pd.qcut(edu_dep[col], q=5, labels=False) + 1


**Above code results in error**

*ValueError: Bin edges must be unique: Index([0.0, 0.0, 0.0, 0.1, 0.3, 1.3], dtype='float64', name='4dim_pct'). You can drop duplicate edges by setting the 'duplicates' kwarg*


---
---

Splitting the 4 Dimensions of Deprivation column into deciles is not possible due to lack of variation between the values.


---
---


Any public good should take into account the most deprived and vulnerable, so although we will proceed with the 0, 1, 2 and 3 dimensions only for the deciles, a binary "4th Dimension" will be considered at the end and for any work based upon this analysis where prioritisation is required, agencies will be able to see where there is an additional dimension of deprivation.

In [ ]:
#using a for loop we can add the deciles only for the deprivation deciles, then we can map the deprivation deciles and see if the more deprived areas are where we would expect them to be.
#start by defining a list of the columns we want to create deciles for
dep_cols=['nodep_pct', '1dim_pct','2dim_pct', '3dim_pct']
#now use a for loop to calculate a new column for each dep_col
for col in dep_cols:
  #replace "pct" in new column names with "dec"
  dep_dec_col=col.replace('_pct', '_dec')
  #calculate 10 bins using qcut
  edu_dep[dep_dec_col]=pd.qcut(edu_dep[col], q=10, labels=False) + 1
#use "where" to add in a yes or no for whether there is additional deprivation
edu_dep['4dim_present']=np.where(edu_dep['4dim_pct']>0, 'Yes', 'No')

#now check and see if the more deprived areas are in the places they are expected to be
#define a list of the decile columns we want to check
decile_cols=['nodep_dec', '1dim_dec', '2dim_dec', '3dim_dec']
#now join the decile cols to the LSOA map
#create a list that'll contain the LSOA code for indexing and then the decile columns defined in the list above
dep_only=edu_dep.loc[:,['lsoacode', 'pop', '4dim_present']+decile_cols]
#use merge to combine dataframes, check resulting dataframe still has 205 rows!
lsoa_dep=nnc_lsoa.merge(dep_only, how='left', left_on='LSOA21CD', right_on='lsoacode')
#lsoa_dep.info()
#create a for loop to plot deciles for the dimensions of deprivation
for i in range (0, len(decile_cols)):
  #create a plot
  fig, ax=plt.subplots(1,1,figsize=(6,6))
  #ensure columns in list are categorical
  lsoa_dep[decile_cols] = lsoa_dep[decile_cols].astype("category")
  #get the relevant item from the list
  lsoa_dep.plot(column=decile_cols[i],
               #format the plot
               linewidth=0.1,
               categorical=True,
               legend=True,
               cmap='RdBu_r',
               ax=ax,
               legend_kwds={'loc': 'center left', 'bbox_to_anchor':(1,0.5), 'title': 'Decile'})
  #add a pink outline to the LSOAs where there are households deprived in four dimensions
  four_dim=lsoa_dep.loc[lsoa_dep['4dim_present']=='Yes']
  four_dim.boundary.plot(linewidth=.5,
                color='magenta',
                alpha=.8,
                ax=ax)
  #add a title based on column, remove axis
  plt.title(decile_cols[i].replace('nodep_dec', 'No Dimension').replace('dim_dec',' Dimensions').capitalize()+' of Deprivation in North Northamptonshire (Decile)')
  plt.axis('off')

The most deprived areas are where we'd expect them to be: In the urban areas, particularly in Corby, Kettering and Wellingborough. Interestingly even in the areas where there is very little other deprivation, there are still some households deprived in four dimensions.

This is likely an effect of the fact that we are working with LSOAs, and especially in more rural areas these are rather large spatial units. If this was run again using OAs instead we would be able to more accurately tell exactly where the deprived households are present.

Now following the same process as above with the education data.

In [ ]:
#using a for loop we can add the deciles for education, then we can map the education deciles and see if the more educated areas are where we would expect them to be.
#start by defining a list of the columns we want to create deciles for
edu_cols=['noqual_pct', 'l1_pct', 'l2_pct', 'apprenticeship_pct', 'l3_pct', 'l4_pct', 'other_pct']
#now use a for loop to calculate a new column for each dep_col
for col in edu_cols:
  #replace "pct" in new column names with "dec"
  ed_dec_col=col.replace('_pct', '_dec')
  #calculate 10 bins using qcut
  edu_dep[ed_dec_col]=pd.qcut(edu_dep[col], q=10, labels=False) + 1
#now check and see if the more deprived areas are in the places they are expected to be
#define a list of the decile columns we want to check
edu_cols=['noqual_dec', 'l1_dec', 'l2_dec', 'apprenticeship_dec', 'l3_dec', 'l4_dec', 'other_dec']
#now join the decile cols to the LSOA map
#create a list that'll contain the LSOA code for indexing and then the decile columns defined in the list above
edu_only=edu_dep.loc[:,['lsoacode', 'pop']+edu_cols]
#use merge to combine dataframes, check resulting dataframe still has 205 rows!
lsoa_edu=nnc_lsoa.merge(edu_only, how='left', left_on='LSOA21CD', right_on='lsoacode')
#lsoa_dep.info()
#create a for loop to plot deciles for the dimensions of deprivation
for i in range (0, len(edu_cols)):
  #create a plot
  fig, ax=plt.subplots(1,1,figsize=(6,6))
  #ensure columns in list are categorical
  lsoa_edu[edu_cols] = lsoa_edu[edu_cols].astype("category")
  #get the relevant item from the list
  lsoa_edu.plot(column=edu_cols[i],
               #format the plot
               linewidth=0.1,
               categorical=True,
               legend=True,
               cmap='RdBu_r',
               ax=ax,
               legend_kwds={'loc': 'center left', 'bbox_to_anchor':(1,0.5), 'title': ' Decile'})
  #add a title based on column, remove axis
  plt.title(edu_cols[i].replace('noqual', 'No Qualification').replace('l1', 'Level 1').replace('l2', 'Level 2').replace('l3', 'Level 3').replace('l4', 'Level 4').replace('_dec', ' % (Decile)').capitalize())
  plt.axis('off')

At a glance it appears that the areas with the highest prevalence of deprivation are those with either no or lower levels of education. Now we'll combine those dataframes and examine the relationships further.

# Analysing the Decile Data

Start by combining the decile data

In [58]:
#merge the dataframes created for the decile data
edu_only.merge(dep_only, 'left', on='lsoacode').drop('pop_y', axis=1)

,lsoacode,pop_x,noqual_dec,l1_dec,l2_dec,apprenticeship_dec,l3_dec,l4_dec,other_dec,4dim_present,nodep_dec,1dim_dec,2dim_dec,3dim_dec
0,E01027033,1595,1,2,1,2,6,10,2,Yes,8,3,2,5
1,E01027043,1980,3,1,3,1,4,10,4,Yes,8,2,4,2
2,E01027050,1800,2,1,2,1,3,10,3,Yes,8,3,3,2
3,E01026971,1509,9,4,1,8,7,3,8,Yes,4,6,8,7
4,E01026979,1352,4,1,3,8,9,8,1,No,10,1,3,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,E01027336,1335,8,6,6,2,4,4,6,No,3,9,8,7
201,E01027337,1500,6,8,5,10,6,4,2,No,5,5,7,5
202,E01027348,1814,4,3,2,9,5,8,5,No,7,5,5,3
203,E01027354,2020,3,4,3,5,7,9,2,Yes,6,5,4,6


In [ ]:
#create a list only of the decile columns and